# Fine-tune Qwen2.5-7B-Instruct with QLoRA (Unsloth) + step-by-step answer checker

**Why 7B:** in the pilot (same 30 test inputs, graded by hand) Qwen 1.5B got 53% fully correct, 3B 60%, **7B 83% with no training**,
while FLAN-T5-base got about 3 out of 27. Fine-tuning teaches our format, style and difficulty levels; the knowledge comes from the 7B model.

**What this notebook does**
1. QLoRA fine-tune with **Unsloth** (about 2x faster, less memory): 4-bit base, LoRA r=16 on all linear layers, loss only on the MCQ.
   Checkpoints every 25 steps; re-running resumes automatically.
2. A stronger **checker** for every generated MCQ (the pilot's letter-only self-check let wrong keys through):
   - strict format: 4 distinct options, one answer letter
   - **numeric consistency:** if the explanation's final number equals a different option than the key, reject
   - **step-by-step verification:** the model solves its own MCQ with reasoning, without the key, and must reach the same letter
3. Comparison on the **same 30 pilot inputs**: 7B as it is (few-shot) vs 7B fine-tuned, saved as CSVs to grade by hand.

**Run:** Kaggle (GPU T4 x2, Internet ON, dataset added) or Colab (T4, dataset zip in Drive `adaptive-quiz`; checkpoints go to Drive).
Estimated time: about 3 to 5 hours of training + about 1 hour of evaluation (estimate; the log prints seconds per step after a few minutes).

In [ ]:
!pip install -q unsloth
!pip install -q scikit-learn

In [ ]:
# install check: if this cell errors, do Runtime/Session > Restart, then run from this cell again (skip the pip cell)
import importlib.metadata as md_
print({p: md_.version(p) for p in ["unsloth", "torch", "transformers", "peft", "bitsandbytes"]})

In [ ]:
import os
os.environ["CUDA_VISIBLE_DEVICES"] = "0"          # Unsloth trains on one GPU
from unsloth import FastLanguageModel, is_bf16_supported      # import unsloth before transformers
import glob, json, math, random, re, time, zipfile, collections, gc
import numpy as np, pandas as pd, torch
ON_KAGGLE = os.path.exists("/kaggle/input")
if ON_KAGGLE:
    DATA_ROOT, OUT_DIR = "/kaggle/input", "/kaggle/working/qwen7b-qlora"
else:
    from google.colab import drive
    drive.mount("/content/drive")
    DRIVE_DIR = "/content/drive/MyDrive/adaptive-quiz"
    DATA_ROOT, OUT_DIR = "/content/work", f"{DRIVE_DIR}/qwen7b-qlora"
    for z in glob.glob(f"{DRIVE_DIR}/*dataset*.zip"):
        out = os.path.join(DATA_ROOT, os.path.basename(z)[:-4])
        if not os.path.exists(out):
            zipfile.ZipFile(z).extractall(out)
os.makedirs(OUT_DIR, exist_ok=True)
assert torch.cuda.is_available(), "no GPU"
print(torch.cuda.get_device_name(0), "| out:", OUT_DIR)

## 1. Config

In [ ]:
CFG = dict(
    model_name="unsloth/Qwen2.5-7B-Instruct-bnb-4bit",   # Qwen2.5-7B-Instruct, pre-quantized to 4-bit (faster download)
    seed=42, max_len=1024,
    epochs=1, train_limit=None,      # set e.g. 5000 to train on a subset and halve the time
    learning_rate=2e-4, lr_scheduler="cosine", warmup_ratio=0.03,
    per_device_batch=2, grad_accum=16,                   # effective batch 32
    lora_r=16, lora_alpha=16, lora_dropout=0.0,
    save_steps=25, val_samples=200,
    n_cand=4,
    run_baseline=True,      # the "before" test (about 40 min); set False to skip it
)
random.seed(CFG["seed"]); np.random.seed(CFG["seed"]); torch.manual_seed(CFG["seed"])
SYSTEM = ("You are a B.Tech exam setter. From the context, write ONE multiple-choice question of the requested difficulty. "
          "Exactly one option is correct; the three wrong options must be different from each other, plausible and clearly wrong. "
          "Check any calculation before answering. Reply in exactly this format: question: <question> options: (A) <option> "
          "(B) <option> (C) <option> (D) <option> answer: <letter> explanation: <one or two sentences>")

## 2. Data

In [ ]:
def find(split):
    hits = sorted(glob.glob(f"{DATA_ROOT}/**/{split}.jsonl", recursive=True), key=lambda p: ("clean" not in p, p))
    assert hits, f"{split}.jsonl not found under {DATA_ROOT}"; return hits[0]
raw = {s: [json.loads(l) for l in open(find(s), encoding="utf-8")] for s in ["train", "val", "test"]}
print({s: len(v) for s, v in raw.items()})

# the same 30 inputs as the pilot notebook (10 per difficulty, spread over branches)
rng = random.Random(0); PILOT = []
for d in ["easy", "medium", "hard"]:
    pool = [r for r in raw["test"] if r["difficulty"] == d]; rng.shuffle(pool)
    by_b = {}
    for r in pool: by_b.setdefault(r["branch"], []).append(r)
    i = 0
    while sum(1 for t in PILOT if t["difficulty"] == d) < 10:
        b = list(by_b)[i % len(by_b)]
        if by_b[b]: PILOT.append(by_b[b].pop())
        i += 1
print(len(PILOT), "pilot inputs")

In [ ]:
import re

LETTERS = "ABCD"


def build_input(branch, subject, topic, difficulty, context):
    return (f"generate mcq | branch: {branch} | subject: {subject} | "
            f"topic: {topic} | difficulty: {difficulty} | context: {context}")


_PATTERN = re.compile(
    r"question:\s*(?P<q>.+?)\s*options:\s*"
    r"\(A\)\s*(?P<A>.+?)\s*\(B\)\s*(?P<B>.+?)\s*\(C\)\s*(?P<C>.+?)\s*\(D\)\s*(?P<D>.+?)\s*"
    r"answer:\s*(?P<ans>[ABCD])\b\s*(?:explanation:\s*(?P<exp>.*))?$",
    re.S)


def parse_mcq(text):
    """Parse model output into a dict, or return None if the format is broken."""
    m = _PATTERN.search(text.strip())
    if not m:
        return None
    options = [m.group(l).strip() for l in LETTERS]
    norm = [re.sub(r"\s+", " ", o.lower()) for o in options]
    if any(not o for o in options) or len(set(norm)) < 4:
        return None
    return {"question": m.group("q").strip(), "options": options,
            "answer": m.group("ans"), "explanation": (m.group("exp") or "").strip()}

## 3. Model (4-bit) + LoRA

In [ ]:
model, tok = FastLanguageModel.from_pretrained(CFG["model_name"], max_seq_length=CFG["max_len"], load_in_4bit=True, dtype=None)
model = FastLanguageModel.get_peft_model(
    model, r=CFG["lora_r"], lora_alpha=CFG["lora_alpha"], lora_dropout=CFG["lora_dropout"], bias="none",
    target_modules=["q_proj", "k_proj", "v_proj", "o_proj", "gate_proj", "up_proj", "down_proj"],
    use_gradient_checkpointing="unsloth", random_state=CFG["seed"])
if tok.pad_token is None:
    tok.pad_token = tok.eos_token
model.print_trainable_parameters()

def prompt_text(input_text, shots=()):
    m = [{"role": "system", "content": SYSTEM}]
    for s in shots:
        m += [{"role": "user", "content": s["input_text"]}, {"role": "assistant", "content": s["target_text"]}]
    m.append({"role": "user", "content": input_text})
    return tok.apply_chat_template(m, tokenize=False, add_generation_prompt=True)

def encode(r):
    p = tok(prompt_text(r["input_text"]), add_special_tokens=False)["input_ids"]
    t = tok(r["target_text"] + tok.eos_token, add_special_tokens=False)["input_ids"]
    ids = (p + t)[:CFG["max_len"]]
    return {"input_ids": ids, "attention_mask": [1] * len(ids), "labels": ([-100] * len(p) + t)[:CFG["max_len"]]}

from datasets import Dataset
train_rows = raw["train"][:CFG["train_limit"]] if CFG["train_limit"] else raw["train"]
val_rows = random.Random(1).sample(raw["val"], CFG["val_samples"])
ds_train = Dataset.from_list([encode(r) for r in train_rows]).shuffle(seed=CFG["seed"])
ds_val = Dataset.from_list([encode(r) for r in val_rows])
lens = np.array([len(x) for x in ds_train["input_ids"]])
print(f"train {len(ds_train)} | tokens mean {lens.mean():.0f} max {lens.max()} | truncated {(lens >= CFG['max_len']).sum()}")
print("label check:", tok.decode([t for t in ds_train[0]["labels"] if t != -100])[:150])

## 4. Checker (used for both models)

In [ ]:
NUM = r"-?\d+(?:\.\d+)?"
def numeric_conflict(p):
    # explanation's final number equals a different option than the key -> wrong key
    nums = re.findall(NUM, p["explanation"].replace(",", ""))
    if not nums: return False
    last = float(nums[-1])
    hits = [i for i, o in enumerate(p["options"])
            if re.fullmatch(r"\s*" + NUM + r"\s*\S{0,8}\s*", o.replace(",", "")) and
               abs(float(re.findall(NUM, o.replace(",", ""))[0]) - last) < 1e-6]
    return bool(hits) and "ABCD"[hits[0]] != p["answer"]

def verify_prompt(p):
    q = p["question"] + "\n" + "\n".join(f"({l}) {o}" for l, o in zip("ABCD", p["options"]))
    m = [{"role": "user", "content": "Solve this multiple-choice question. Think step by step and check any calculation. "
                                     "If no option or more than one option is correct, say so. "
                                     "End with a last line exactly like: Final answer: <letter or NONE>\n\n" + q}]
    return tok.apply_chat_template(m, tokenize=False, add_generation_prompt=True)

@torch.no_grad()
def lm(prompts, n=1, sample=True, max_new=320):
    tok.padding_side = "left"
    enc = tok(prompts, return_tensors="pt", padding=True, add_special_tokens=False).to("cuda")
    kw = dict(do_sample=True, temperature=0.7, top_p=0.9) if sample else dict(do_sample=False)
    ids = model.generate(**enc, max_new_tokens=max_new, num_return_sequences=n, pad_token_id=tok.pad_token_id, **kw)
    return tok.batch_decode(ids[:, enc["input_ids"].shape[1]:], skip_special_tokens=True)

def generate_checked(rows, shots=(), n_cand=CFG["n_cand"], bs=2):
    out = []
    for i in range(0, len(rows), bs):
        chunk = rows[i:i+bs]
        texts = lm([prompt_text(r["input_text"], shots) for r in chunk], n=n_cand)
        for j, r in enumerate(chunk):
            cands = [p for p in map(parse_mcq, texts[j*n_cand:(j+1)*n_cand]) if p]
            res = {"valid": bool(cands), "passed_numeric": False, "passed_verify": False, "mcq": None, "verify_note": ""}
            cands_num = [p for p in cands if not numeric_conflict(p)]
            res["passed_numeric"] = bool(cands_num)
            if cands_num:
                ver = lm([verify_prompt(p) for p in cands_num], sample=False, max_new=400)
                for p, v in zip(cands_num, ver):
                    m = re.search(r"Final answer:\s*\(?([A-D]|NONE)", v)
                    if m and m.group(1) == p["answer"]:
                        res.update(passed_verify=True, mcq=p, verify_note=v[-300:]); break
                if not res["mcq"]:
                    res.update(mcq=cands_num[0], verify_note=ver[0][-300:])
            elif cands:
                res["mcq"] = cands[0]
            out.append(res)
    return out

def to_rows(rows, gens, label):
    recs = []
    for r, g in zip(rows, gens):
        p = g["mcq"]
        recs.append({"model": label, "branch": r["branch"], "subject": r["subject"], "topic": r["topic"],
                     "difficulty": r["difficulty"], "valid": g["valid"], "passed_numeric": g["passed_numeric"],
                     "passed_verify": g["passed_verify"],
                     "mcq": None if not p else f"Q: {p['question']}\n" + "\n".join(f"  ({l}) {o}" for l, o in zip("ABCD", p["options"]))
                            + f"\n  answer: {p['answer']}  |  {p['explanation']}",
                     "verify_note": g["verify_note"], "is_correct (you fill Y/N)": ""})
    return recs

## 5. Baseline: 7B as it is (few-shot, adapters off) on the 30 pilot inputs

In [ ]:
SHOTS = [next(r for r in raw["train"] if r["difficulty"] == "easy" and r["branch"] == "CSE"),
         next(r for r in raw["train"] if r["difficulty"] == "hard" and r["branch"] != "CSE")]
base_csv = f"{OUT_DIR}/eval_base7b_pilot30.csv"
prev = glob.glob("/kaggle/input/**/eval_base7b_pilot30.csv", recursive=True)
if prev and not os.path.exists(base_csv):
    import shutil; shutil.copy(prev[0], base_csv); print("reusing baseline from", prev[0])
if not os.path.exists(base_csv) and CFG["run_baseline"]:
    FastLanguageModel.for_inference(model)
    t0 = time.time()
    try:
        with model.disable_adapter():
            base_gens = generate_checked(PILOT, shots=SHOTS)
        pd.DataFrame(to_rows(PILOT, base_gens, "Qwen2.5-7B few-shot (no training)")).to_csv(base_csv, index=False)
        print(f"baseline done in {(time.time()-t0)/60:.1f} min")
    except Exception as e:      # the pilot already measured 7B without training, so training can go on
        print("baseline skipped:", repr(e)[:300])
    FastLanguageModel.for_training(model)
if os.path.exists(base_csv):
    print(pd.read_csv(base_csv)[["valid", "passed_numeric", "passed_verify"]].mean().round(3))

## 6. Train (resumes automatically)

In [ ]:
from transformers import Trainer, TrainingArguments, DataCollatorForSeq2Seq, TrainerCallback
class PrintProgress(TrainerCallback):
    def on_log(self, args, state, control, logs=None, **kw):
        if logs:
            print(f"[{time.strftime('%H:%M:%S')}] step {state.global_step}/{state.max_steps} "
                  + " ".join(f"{k}={v:.4g}" if isinstance(v, float) else f"{k}={v}" for k, v in logs.items()), flush=True)
class ShowSamples(TrainerCallback):
    # at every evaluation, print 2 MCQs for fixed val inputs, so you can judge quality while it trains
    def on_evaluate(self, args, state, control, **kw):
        try:
            model.eval()
            for r in val_rows[:2]:
                print(f"   sample [{r['difficulty']}] {r['topic']}:", lm([prompt_text(r["input_text"])], sample=False, max_new=200)[0], flush=True)
        except Exception as e:
            print("   (sample skipped:", repr(e)[:150], ")")
        finally:
            tok.padding_side = "right"; model.train()

tok.padding_side = "right"
import inspect
total_steps = math.ceil(len(ds_train) / (CFG["per_device_batch"] * CFG["grad_accum"])) * CFG["epochs"]
targs = dict(
    output_dir=OUT_DIR, num_train_epochs=CFG["epochs"], learning_rate=CFG["learning_rate"],
    lr_scheduler_type=CFG["lr_scheduler"], warmup_steps=math.ceil(CFG["warmup_ratio"] * total_steps),  # int works in every version
    per_device_train_batch_size=CFG["per_device_batch"], per_device_eval_batch_size=CFG["per_device_batch"],
    gradient_accumulation_steps=CFG["grad_accum"], fp16=not is_bf16_supported(), bf16=is_bf16_supported(),
    optim="adamw_8bit", weight_decay=0.0, max_grad_norm=1.0,
    eval_strategy="steps", eval_steps=CFG["save_steps"] * 2, save_strategy="steps", save_steps=CFG["save_steps"],
    save_total_limit=3, logging_steps=5, report_to="none", seed=CFG["seed"],
)
# check every argument against the installed transformers version before 5 hours of training
accepted = set(inspect.signature(TrainingArguments.__init__).parameters)
missing = [k for k in targs if k not in accepted]
assert not missing, f"transformers {__import__('transformers').__version__} does not accept: {missing}"
args = TrainingArguments(**targs)
print("total optimizer steps:", total_steps, "| warmup steps:", targs["warmup_steps"])
trainer = Trainer(model=model, args=args, train_dataset=ds_train, eval_dataset=ds_val,
                  data_collator=DataCollatorForSeq2Seq(tok, padding=True, label_pad_token_id=-100),
                  callbacks=[PrintProgress(), ShowSamples()])
ck = sorted(glob.glob(f"{OUT_DIR}/checkpoint-*"), key=lambda p: int(p.split("-")[-1]))
print("resuming from", ck[-1] if ck else None)
t0 = time.time()
trainer.train(resume_from_checkpoint=ck[-1] if ck else None)
print(f"training took {(time.time()-t0)/60:.1f} min")
ADAPTER_DIR = f"{OUT_DIR}/final_adapter"
model.save_pretrained(ADAPTER_DIR); tok.save_pretrained(ADAPTER_DIR)
json.dump({**CFG, "system": SYSTEM}, open(f"{ADAPTER_DIR}/train_config.json", "w"), indent=2)
print("saved adapter to", ADAPTER_DIR)

In [ ]:
import matplotlib.pyplot as plt
h = pd.DataFrame(trainer.state.log_history)
tr, ev = h.dropna(subset=["loss"]), h.dropna(subset=["eval_loss"]) if "eval_loss" in h else h.iloc[:0]
plt.figure(figsize=(7, 4)); plt.plot(tr.step, tr.loss, label="train")
if len(ev): plt.plot(ev.step, ev.eval_loss, "o-", label="val")
plt.xlabel("step"); plt.title("Qwen2.5-7B QLoRA loss"); plt.legend(); plt.savefig(f"{OUT_DIR}/training_curves.png", dpi=120); plt.show()

## 7. Fine-tuned 7B on the same 30 pilot inputs + 60 more test inputs

In [ ]:
FastLanguageModel.for_inference(model)
extra = [r for r in random.Random(5).sample(raw["test"], 120) if r not in PILOT][:60]
t0 = time.time()
ft_gens = generate_checked(PILOT + extra)
print(f"generated {len(ft_gens)} in {(time.time()-t0)/60:.1f} min")
ft = pd.DataFrame(to_rows(PILOT + extra, ft_gens, "Qwen2.5-7B QLoRA fine-tuned"))
ft.iloc[:30].to_csv(f"{OUT_DIR}/eval_ft7b_pilot30.csv", index=False)
ft.to_csv(f"{OUT_DIR}/eval_ft7b_all90.csv", index=False)

In [ ]:
from sklearn.feature_extraction.text import TfidfVectorizer
from sklearn.linear_model import LogisticRegression
from sklearn.pipeline import make_pipeline
def mcq_text(p): return p["question"] + " || " + " | ".join(p["options"])
trp = [(parse_mcq(r["target_text"]), r["difficulty"]) for r in raw["train"]]
clf = make_pipeline(TfidfVectorizer(ngram_range=(1, 2), min_df=2, sublinear_tf=True), LogisticRegression(max_iter=2000, C=2.0))
clf.fit([mcq_text(p) for p, _ in trp if p], [d for p, d in trp if p])
rows = PILOT + extra
ok = [(g["mcq"], r) for g, r in zip(ft_gens, rows) if g["mcq"]]
base = pd.read_csv(base_csv) if os.path.exists(base_csv) else pd.DataFrame({"valid": [np.nan], "passed_numeric": [np.nan], "passed_verify": [np.nan], "mcq": [None]})
summary = pd.DataFrame([
    {"model": "7B few-shot (pilot 30)", "valid_%": 100 * base.valid.mean(), "passed_numeric_%": 100 * base.passed_numeric.mean(),
     "passed_verify_%": 100 * base.passed_verify.mean()},
    {"model": "7B fine-tuned (pilot 30)", **{k: 100 * ft.iloc[:30][c].mean() for k, c in
        [("valid_%", "valid"), ("passed_numeric_%", "passed_numeric"), ("passed_verify_%", "passed_verify")]}},
    {"model": "7B fine-tuned (all 90)", **{k: 100 * ft[c].mean() for k, c in
        [("valid_%", "valid"), ("passed_numeric_%", "passed_numeric"), ("passed_verify_%", "passed_verify")]},
     "difficulty_match_%": 100 * np.mean(clf.predict([mcq_text(p) for p, _ in ok]) == np.array([r["difficulty"] for _, r in ok])),
     "answer_A/B/C/D_%": "/".join(str(round(100 * collections.Counter(p["answer"] for p, _ in ok)[l] / len(ok))) for l in "ABCD")},
]).round(1)
summary.to_csv(f"{OUT_DIR}/summary_7b.csv", index=False); print(summary.to_string(index=False))

## 8. Side by side: same input, 7B before and after fine-tuning

In [ ]:
for k, t in enumerate(PILOT[:12]):
    print("#" * 100, f"\n[{t['difficulty']}] {t['branch']} | {t['subject']} | {t['topic']}")
    for name, df_ in [("BEFORE (few-shot)", base), ("AFTER (fine-tuned)", ft)]:
        if k >= len(df_): continue
        r = df_.iloc[k]
        print(f"--- {name}  verify: {'PASS' if r.passed_verify else 'fail'}")
        print(r.mcq if isinstance(r.mcq, str) else "NO VALID MCQ")

## 9. Your own input

In [ ]:
BRANCH, SUBJECT, TOPIC = "CSE", "Operating Systems", "Deadlock"
CONTEXT = """A deadlock is a situation where a set of processes are blocked because each process is holding a resource
and waiting for another resource held by some other process. Four conditions must hold at the same time for a deadlock:
mutual exclusion, hold and wait, no preemption and circular wait. Deadlocks can be handled by prevention, which breaks one
of the four conditions, by avoidance, such as the Banker's algorithm, which grants a request only if the system stays in a
safe state, or by detection and recovery, where the system finds cycles in the wait-for graph and then aborts a process or
preempts resources."""
mine = [{"input_text": build_input(BRANCH, SUBJECT, TOPIC, d, " ".join(CONTEXT.split())), "difficulty": d} for d in ["easy", "medium", "hard"]]
for r, g in zip(mine, generate_checked(mine)):
    p = g["mcq"]; print(f"\n[{r['difficulty']}] verify: {'PASS' if g['passed_verify'] else 'fail'}")
    if p:
        print(p["question"]); [print(f"   ({l}) {o}") for l, o in zip("ABCD", p["options"])]
        print("   correct:", p["answer"], "->", p["options"]["ABCD".index(p["answer"])], "|", p["explanation"])

## 10. Zip the adapter (about 160 MB) for the app

In [ ]:
import shutil
shutil.make_archive(f"{OUT_DIR}/qwen7b_mcq_adapter", "zip", ADAPTER_DIR)
print(round(os.path.getsize(f"{OUT_DIR}/qwen7b_mcq_adapter.zip") / 1e6), "MB")